In [1]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import netCDF4 as n
import glob
import xarray as xr
import seaborn as sns
import matplotlib.lines as mlines
import pandas as pd
import function_sampling_Thermal_forcing_files as fn
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.stats import linregress

from collections import Counter
sns.set_context("paper")
sns.set_style("whitegrid")
computed_name = 'ComputedScalarsHelene'


In [2]:
 
mnt_pth = '/Volumes/CrucialX8/computing/data/antarctica/' #mount path
pth_imip6hack= mnt_pth + 'ismip6_hackathon/ComputedScalars/' #write folder
pth_ismip6 =mnt_pth + 'ismip6_hackathon/' #read folder
pth_helene =mnt_pth + 'ismip6_hackathon/ComputedScalarsHelene/'
 
expnames = ['expAE02', 'expAE03', 'expAE04', 'expAE05']
removeFileID = [] # specify models to remove
 
metadata_file = 'Metadata.txt'
# Generate loop_info DataFrame

In [8]:
def get_model(string,exp):
    return string.split('/')[-1].split('AIS_')[-1].split('_'+str(exp))[0]
dic_area_of_interest = {}
dic_area_of_interest['FilchnerRonne'] = ['_sector_5','_sector_11']
dic_area_of_interest['Ross'] = ['_sector_2','_sector_6']
dic_area_of_interest['Amundsen'] = ['_sector_4']
y2sec=-365.25*24*3600
factor = y2sec/10**12

In [9]:
# avg and abs 2100
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
for key in dic_area_of_interest.keys():
    df[key]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    #get shelf area
    pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/iareafl/computed_iareafl_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    
    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )
    #
    ind = np.where(d.time.values <= 2100)
    
    
    
    tmin = np.min([d.time.size,d2.time.size])
    #melting per area
    y =d.shelfmelt.values[:tmin]* factor/d2.iareafl.values[:tmin] #T/m^3 yr
    yBMB =d.shelfmelt.values[:tmin]* factor
    
   
    yais =y[ind].mean()
    yaisBMB =yBMB[ind].cumsum()[-1]
    
    
    df['AIS'].iloc[i]=  yais
    df2['AIS'].iloc[i]=  yaisBMB
    

    for j in range(1,19):
        y =d['shelfmelt_sector_'+str(j)].values[:tmin]* factor/d2['iareafl_sector_'+str(j)].values[:tmin]  #GT/m^3 yr
        yBMB=d['shelfmelt_sector_'+str(j)].values[:tmin]* factor
        
        mnan = np.isnan(y)
        
        
       
        df['sector_'+str(j)].iloc[i] = y[ind].mean()
        df2['sector_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
    for j in range(1,4):
        y =d['shelfmelt_region_'+str(j)].values[:tmin]* factor/d2['iareafl_region_'+str(j)].values[:tmin]  #GT/m^3 yr
        yBMB=d['shelfmelt_region_'+str(j)].values[:tmin]* factor
        
        mnan = np.isnan(y)
        
        
       
        df['region_'+str(j)].iloc[i] = y[ind].mean()
        df2['region_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
        
    for key in dic_area_of_interest.keys():
        sectors = dic_area_of_interest[key].copy()
        ys = np.zeros((len(sectors),tmin))
        ysa = np.zeros((len(sectors),tmin))
        
        areas = np.zeros((len(sectors),tmin))
    
        for j,sector in enumerate(sectors):
            ys[j,:] = d[f'shelfmelt{sector}'].values[:tmin].copy()* factor
            areas[j,:] = d2[f'iareafl{sector}'].values[:tmin].copy()
            ysa[j,:]=ys[j,:]*areas[j,:]
        yBMB =  np.nansum(ys,axis = 0)
        y =  np.nansum(ys,axis = 0)/np.nansum(areas,axis= 0)
        df[key].iloc[i] = y[ind].mean()
        df2[key].iloc[i] = yBMB[ind].cumsum()[-1]
        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
# df.to_csv('tables/averegeshelfmelt_2100.csv', index=False)
df2.to_csv('tables/cumulative_shelfmelt_2100.csv', index=False)

/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_9740/2671604305.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]=  yais
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_9740/2671604305.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['AIS'].iloc[i]=  yaisBMB
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_9740/2671604305.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  d

In [10]:
df2

,Experiment,Model,Path,sector_1,sector_2,sector_3,sector_4,sector_5,sector_6,sector_7,...,sector_16,sector_17,sector_18,region_1,region_2,region_3,FilchnerRonne,Ross,Amundsen,AIS
0,expAE02,DC_ISSM,/Volumes/CrucialX8/computing/data/antarctica/i...,5906.360840,15960.875977,5712.744141,16371.270508,73978.656250,9639.600586,2612.910889,...,1427.904663,4282.601562,10795.021484,117929.898438,128796.062500,16505.533203,131070.422766,25600.480502,16371.270248,263231.53125
1,expAE02,DOE_MALI_4km,/Volumes/CrucialX8/computing/data/antarctica/i...,8094.968262,21649.365234,16430.388672,28086.404297,100839.968750,11227.783203,3285.675537,...,1074.078613,5617.290527,12837.619141,175101.109375,161089.281250,19528.986328,173716.218357,32877.149911,28086.399399,355719.34375
2,expAE02,DOE_MALI_8km_Ant95,/Volumes/CrucialX8/computing/data/antarctica/i...,7844.681641,26562.603516,15170.358398,26709.480469,119268.820312,13467.635742,2412.543213,...,1294.443604,5318.756348,11863.493164,195555.921875,181435.093750,18476.693359,206716.951649,40030.243602,26709.479446,395467.71875
3,expAE02,DOE_MALI_8km_AntMean,/Volumes/CrucialX8/computing/data/antarctica/i...,7456.148438,21199.250000,14611.526367,25763.201172,99963.078125,11208.141602,2185.497070,...,1146.996826,5049.943848,11640.058594,168993.218750,154661.687500,17837.001953,172474.453846,32407.388492,25763.204483,341491.84375
4,expAE02,IGE_ElmerIce,/Volumes/CrucialX8/computing/data/antarctica/i...,7540.280273,13844.632812,14533.190430,18621.474609,36119.332031,2584.670654,3386.814209,...,4791.292480,9896.978516,13678.037109,90658.906250,114470.976562,28366.306641,63910.777607,16429.302946,18621.479233,233496.18750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167,expAE05,VUW_PISM2,/Volumes/CrucialX8/computing/data/antarctica/i...,21429.113281,99465.640625,32739.009766,22879.000000,21427.914062,31996.902344,5752.481445,...,-1976.970703,-3450.101318,17808.304688,197940.656250,169943.375000,12381.236328,40006.699370,131462.549297,22879.000278,380265.21875
168,expAE05,VUW_PISM2_s1,/Volumes/CrucialX8/computing/data/antarctica/i...,23534.875000,100249.398438,33751.410156,24555.794922,20864.566406,32244.847656,9772.541016,...,-1753.602539,-3096.356934,18081.246094,202956.078125,191470.375000,13231.285156,39509.613276,132494.261433,24555.795668,407657.81250
169,expAE05,VUW_PISM2_s2,/Volumes/CrucialX8/computing/data/antarctica/i...,19949.716797,98240.078125,32876.691406,22470.646484,21795.378906,32116.175781,7171.635254,...,-1763.423706,-3321.138672,17607.037109,195332.453125,166765.531250,12522.480469,40768.593131,130356.244617,22470.645596,374620.50000
170,expAE05,VUW_PISM2_s3,/Volumes/CrucialX8/computing/data/antarctica/i...,22802.308594,99714.976562,32547.921875,24031.228516,21905.406250,32043.753906,6467.788574,...,-1956.851807,-3594.310303,18416.898438,201001.906250,173869.062500,12865.732422,40624.517475,131758.765161,24031.227705,387736.75000


In [11]:
# avg and abs 2100
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
for key in dic_area_of_interest.keys():
    df[key]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    #get shelf area
    pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/iareafl/computed_iareafl_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    
    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )
    #
    ind = np.where(d.time.values <= 2200)
    
    
    
    tmin = np.min([d.time.size,d2.time.size])
    #melting per area
    y =d.shelfmelt.values[:tmin]* factor/d2.iareafl.values[:tmin] #T/m^3 yr
    yBMB =d.shelfmelt.values[:tmin]* factor
    
   
    yais =y[ind].mean()
    yaisBMB =yBMB[ind].cumsum()[-1]
    
    
    df['AIS'].iloc[i]=  yais
    df2['AIS'].iloc[i]=  yaisBMB
    

    for j in range(1,19):
        y =d['shelfmelt_sector_'+str(j)].values[:tmin]* factor/d2['iareafl_sector_'+str(j)].values[:tmin]  #GT/m^3 yr
        yBMB=d['shelfmelt_sector_'+str(j)].values[:tmin]* factor
        
        mnan = np.isnan(y)
        
        
       
        df['sector_'+str(j)].iloc[i] = y[ind].mean()
        df2['sector_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
    for j in range(1,4):
        y =d['shelfmelt_region_'+str(j)].values[:tmin]* factor/d2['iareafl_region_'+str(j)].values[:tmin]  #GT/m^3 yr
        yBMB=d['shelfmelt_region_'+str(j)].values[:tmin]* factor
        
        mnan = np.isnan(y)
        
        
       
        df['region_'+str(j)].iloc[i] = y[ind].mean()
        df2['region_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
        
    for key in dic_area_of_interest.keys():
        sectors = dic_area_of_interest[key].copy()
        ys = np.zeros((len(sectors),tmin))
        ysa = np.zeros((len(sectors),tmin))
        
        areas = np.zeros((len(sectors),tmin))
    
        for j,sector in enumerate(sectors):
            ys[j,:] = d[f'shelfmelt{sector}'].values[:tmin].copy()* factor
            areas[j,:] = d2[f'iareafl{sector}'].values[:tmin].copy()
            ysa[j,:]=ys[j,:]*areas[j,:]
        yBMB =  np.nansum(ys,axis = 0)
        y =  np.nansum(ys,axis = 0)/np.nansum(areas,axis= 0)
        df[key].iloc[i] = y[ind].mean()
        df2[key].iloc[i] = yBMB[ind].cumsum()[-1]
        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
# df.to_csv('tables/averegeshelfmelt_2100.csv', index=False)
df2.to_csv('tables/cumulative_shelfmelt_2200.csv', index=False)

/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_9740/1022863249.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]=  yais
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_9740/1022863249.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['AIS'].iloc[i]=  yaisBMB
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_9740/1022863249.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  d

In [12]:
# avg and abs 2100
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
for key in dic_area_of_interest.keys():
    df[key]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    #get shelf area
    pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/iareafl/computed_iareafl_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    
    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )
    #
    ind = np.where(d.time.values <= 2300)
    
    
    
    tmin = np.min([d.time.size,d2.time.size])
    #melting per area
    y =d.shelfmelt.values[:tmin]* factor/d2.iareafl.values[:tmin] #T/m^3 yr
    yBMB =d.shelfmelt.values[:tmin]* factor
    
   
    yais =y[ind].mean()
    yaisBMB =yBMB[ind].cumsum()[-1]
    
    
    df['AIS'].iloc[i]=  yais
    df2['AIS'].iloc[i]=  yaisBMB
    

    for j in range(1,19):
        y =d['shelfmelt_sector_'+str(j)].values[:tmin]* factor/d2['iareafl_sector_'+str(j)].values[:tmin]  #GT/m^3 yr
        yBMB=d['shelfmelt_sector_'+str(j)].values[:tmin]* factor
        
        mnan = np.isnan(y)
        
        
       
        df['sector_'+str(j)].iloc[i] = y[ind].mean()
        df2['sector_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
    for j in range(1,4):
        y =d['shelfmelt_region_'+str(j)].values[:tmin]* factor/d2['iareafl_region_'+str(j)].values[:tmin]  #GT/m^3 yr
        yBMB=d['shelfmelt_region_'+str(j)].values[:tmin]* factor
        
        mnan = np.isnan(y)
        
        
       
        df['region_'+str(j)].iloc[i] = y[ind].mean()
        df2['region_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
        
    for key in dic_area_of_interest.keys():
        sectors = dic_area_of_interest[key].copy()
        ys = np.zeros((len(sectors),tmin))
        ysa = np.zeros((len(sectors),tmin))
        
        areas = np.zeros((len(sectors),tmin))
    
        for j,sector in enumerate(sectors):
            ys[j,:] = d[f'shelfmelt{sector}'].values[:tmin].copy()* factor
            areas[j,:] = d2[f'iareafl{sector}'].values[:tmin].copy()
            ysa[j,:]=ys[j,:]*areas[j,:]
        yBMB =  np.nansum(ys,axis = 0)
        y =  np.nansum(ys,axis = 0)/np.nansum(areas,axis= 0)
        df[key].iloc[i] = y[ind].mean()
        df2[key].iloc[i] = yBMB[ind].cumsum()[-1]
        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
# df.to_csv('tables/averegeshelfmelt_2100.csv', index=False)
df2.to_csv('tables/cumulative_shelfmelt_2300.csv', index=False)

/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_9740/92400511.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]=  yais
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_9740/92400511.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['AIS'].iloc[i]=  yaisBMB
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_9740/92400511.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sec